# **Adaptive Agentic RAG: Reinforcement Learning for Retrieval, Query Rewriting, and Answer Routing**

## 1. INSTALLATION

In [1]:
!pip -q install -U \
    "langgraph>=0.2,<2" \
    "langchain-text-splitters>=0.3" \
    "sentence-transformers>=3" \
    "faiss-cpu>=1.8" \
    "stable-baselines3>=2.3" \
    "gymnasium>=0.29" \
    "transformers>=4.45" \
    "accelerate>=1.0" \
    "safetensors>=0.4"

print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.9 MB/s eta 0:00:00
Installation complete.


## 2. Imports, device detection and reproducibility

The LLM is downloaded from Hugging Face and executed locally; no OpenAI, Gemini, Groq or other paid inference API is required. Transformers provides local model inference through its model/pipeline APIs. citeturn0search1turn0search6

In [2]:
import os, re, gc, json, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("CPU mode: functional but slower.")

PyTorch: 2.11.0+cpu
Device: cpu
CPU mode: functional but slower.


## 3. Persian knowledge base

The first prototype had a very small KB. 

```mermaid
flowchart LR
    A[📄 Persian<br/>Documents] --> B[✂️ Chunking<br/>450 / 70]
    B --> C[📦 Chunks<br/>16 items]
    C --> D[(🗄️ Metadata<br/>doc_id · title · keywords)]

    style A fill:#e3f2fd,stroke:#1976d2,stroke-width:2px,color:#0d47a1
    style B fill:#fff3e0,stroke:#f57c00,stroke-width:2px,color:#e65100
    style C fill:#e8f5e9,stroke:#388e3c,stroke-width:2px,color:#1b5e20
    style D fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px,color:#4a148c
```

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCUMENTS = [
    {"id":"dl","title":"یادگیری عمیق","text":"یادگیری عمیق شاخه‌ای از یادگیری ماشین است که از شبکه‌های عصبی چندلایه برای یادگیری نمایش‌های پیچیده استفاده می‌کند. شبکه‌های کانولوشنی برای بینایی ماشین و ترنسفورمرها برای داده‌های ترتیبی کاربرد دارند.","keywords":["یادگیری عمیق","شبکه عصبی","یادگیری ماشین","CNN","ترنسفورمر"]},
    {"id":"cnn","title":"شبکه عصبی کانولوشنی","text":"شبکه عصبی کانولوشنی یا CNN برای پردازش داده‌های شبکه‌ای مانند تصویر طراحی شده است. کانولوشن، pooling و لایه‌های fully connected از اجزای رایج آن هستند. CNN در طبقه‌بندی تصویر، تشخیص شیء و کنترل کیفیت صنعتی استفاده می‌شود.","keywords":["CNN","کانولوشن","تصویر","تشخیص شیء","pooling"]},
    {"id":"transformer","title":"ترنسفورمر","text":"معماری Transformer بر مکانیزم attention استوار است و امکان پردازش موازی توالی را فراهم می‌کند. مدل‌هایی مانند BERT و خانواده GPT از این معماری استفاده می‌کنند. attention وابستگی میان بخش‌های مختلف ورودی را مدل می‌کند.","keywords":["ترنسفورمر","attention","BERT","GPT","NLP"]},
    {"id":"rag","title":"بازیابی تقویت‌شده با تولید","text":"RAG یا Retrieval-Augmented Generation پاسخ‌گویی مدل زبانی را با بازیابی اسناد خارجی ترکیب می‌کند. ابتدا پرسش به بردار تبدیل و اسناد مرتبط بازیابی می‌شوند؛ سپس مدل زبانی با استفاده از context پاسخ تولید می‌کند. این روش می‌تواند خطای اطلاعاتی را کاهش دهد.","keywords":["RAG","بازیابی","مدل زبانی","context","hallucination"]},
    {"id":"retrieval","title":"بازیابی اطلاعات","text":"در سیستم‌های بازیابی اطلاعات، پرسش با اسناد مقایسه می‌شود و مرتبط‌ترین اسناد بر اساس شباهت بازگردانده می‌شوند. embedding، cosine similarity، FAISS، precision و recall از مفاهیم مهم بازیابی هستند.","keywords":["بازیابی","embedding","FAISS","شباهت","recall","precision"]},
    {"id":"agents","title":"عامل‌های هوشمند","text":"Agent می‌تواند با مشاهده وضعیت، انتخاب عمل، دریافت نتیجه و ادامه مسیر مسئله را به صورت چندمرحله‌ای حل کند. در معماری‌های agentic RAG، عامل می‌تواند بین بازیابی، بازنویسی پرسش و پاسخ‌گویی تصمیم بگیرد.","keywords":["agent","agentic RAG","routing","بازنویسی","تصمیم‌گیری"]},
    {"id":"rl","title":"یادگیری تقویتی","text":"در یادگیری تقویتی، عامل در یک محیط عمل انتخاب می‌کند و بر اساس پاداش یاد می‌گیرد. وضعیت، action، reward و policy عناصر اصلی هستند. هدف، بیشینه‌کردن بازده تجمعی مورد انتظار است.","keywords":["یادگیری تقویتی","RL","reward","policy","action"]},
    {"id":"ppo","title":"الگوریتم PPO","text":"PPO یا Proximal Policy Optimization یک الگوریتم policy-gradient است که با محدود کردن تغییرات policy به پایداری آموزش کمک می‌کند. پارامترهایی مانند clip range، learning rate و entropy coefficient در آموزش PPO مهم هستند.","keywords":["PPO","policy gradient","clip","policy","reinforcement learning"]},
    {"id":"recommendation","title":"سیستم توصیه‌گر","text":"سیستم‌های توصیه‌گر با استفاده از تعاملات کاربر و ویژگی‌های اقلام، پیشنهاد شخصی‌سازی‌شده تولید می‌کنند. collaborative filtering، content-based filtering و روش‌های hybrid از رویکردهای رایج هستند.","keywords":["سیستم توصیه‌گر","collaborative filtering","content-based","hybrid"]},
    {"id":"cv","title":"بینایی ماشین صنعتی","text":"بینایی ماشین صنعتی برای بازرسی کیفیت، شمارش قطعات، تشخیص عیب و کنترل خط تولید استفاده می‌شود. دوربین، مدل تشخیص شیء، پردازش تصویر و سیستم تصمیم‌گیری اجزای رایج چنین سامانه‌ای هستند.","keywords":["بینایی ماشین","کنترل کیفیت","تشخیص عیب","خط تولید","YOLO"]},
    {"id":"nlp","title":"پردازش زبان طبیعی","text":"پردازش زبان طبیعی یا NLP شامل روش‌هایی برای تحلیل و تولید زبان انسان است. طبقه‌بندی متن، استخراج اطلاعات، ترجمه، خلاصه‌سازی و پاسخ‌گویی به پرسش از کاربردهای آن هستند.","keywords":["NLP","پردازش زبان طبیعی","خلاصه‌سازی","طبقه‌بندی","ترجمه"]},
    {"id":"llm","title":"مدل‌های زبانی بزرگ","text":"LLMها مدل‌های زبانی بزرگ هستند که با داده‌های متنی وسیع آموزش می‌بینند. تولید متن، پاسخ به پرسش، خلاصه‌سازی و استفاده از ابزار از کاربردهای آنهاست. کیفیت prompt و context بر خروجی اثر دارد.","keywords":["LLM","مدل زبانی بزرگ","prompt","context","تولید متن"]},
    {"id":"embeddings","title":"Embedding","text":"Embedding متن را به یک بردار عددی در فضای چندبعدی تبدیل می‌کند تا شباهت معنایی میان متن‌ها قابل محاسبه شود. مدل‌های multilingual برای زبان فارسی کاربرد دارند و می‌توان آنها را با FAISS جست‌وجو کرد.","keywords":["embedding","بردار","شباهت معنایی","multilingual","FAISS"]},
    {"id":"vector-db","title":"پایگاه برداری","text":"Vector database یا index برای ذخیره و جست‌وجوی بردارها طراحی شده است. FAISS کتابخانه‌ای سریع برای similarity search است و برای ساخت نمونه‌های RAG محلی گزینه مناسبی محسوب می‌شود.","keywords":["vector database","FAISS","similarity search","بردار"]},
    {"id":"hallucination","title":"توهم مدل زبانی","text":"Hallucination زمانی رخ می‌دهد که مدل زبانی اطلاعات نادرست یا بدون پشتوانه تولید کند. grounding با اسناد، citation و ارزیابی faithfulness از روش‌های کاهش این مشکل هستند.","keywords":["hallucination","faithfulness","grounding","citation"]},
    {"id":"evaluation","title":"ارزیابی RAG","text":"ارزیابی RAG باید retrieval و generation را جداگانه بررسی کند. Recall@K، Precision@K، MRR، context relevance، answer correctness و faithfulness معیارهای مهمی هستند.","keywords":["evaluation","Recall@K","Precision@K","MRR","faithfulness"]},
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=70,
    separators=["\n\n","\n","؟","،","."," "]
)

CHUNKS = []
for d in DOCUMENTS:
    parts = splitter.split_text(d["text"])
    for j, part in enumerate(parts):
        CHUNKS.append({
            "chunk_id": f'{d["id"]}_{j}',
            "doc_id": d["id"],
            "title": d["title"],
            "text": part,
            "keywords": d["keywords"]
        })

print("Documents:", len(DOCUMENTS))
print("Chunks:", len(CHUNKS))

Documents: 16
Chunks: 16


## 4. Multilingual embeddings + FAISS

## 🏗️ System Architecture

```mermaid
flowchart TB
    Q([❓ User Query]) --> R{{🧠 PPO Router}}

    R -->|Action 1| RET[🔍 Retrieve]
    R -->|Action 2| RW[✏️ Rewrite]
    R -->|Action 3| RA[🔁 Retrieve Again]
    R -->|Action 4| ANS[💬 Generate Answer]

    RET --> KB[(📚 FAISS KB)]
    RW --> NQ[🆕 New Query]
    NQ --> KB
    RA --> KB
    KB --> CTX[📄 Retrieved Context]
    CTX --> LLM[🤖 Local LLM<br/>Qwen2.5-0.5B]
    ANS --> LLM
    LLM --> OUT([✅ Final Answer])

    style Q fill:#bbdefb,stroke:#1565c0,stroke-width:2px,color:#0d47a1
    style R fill:#ffe0b2,stroke:#ef6c00,stroke-width:2px,color:#e65100
    style RET fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20
    style RW fill:#ffecb3,stroke:#ff8f00,stroke-width:2px,color:#e65100
    style RA fill:#b2dfdb,stroke:#00796b,stroke-width:2px,color:#004d40
    style ANS fill:#d1c4e9,stroke:#5e35b1,stroke-width:2px,color:#311b92
    style KB fill:#f8bbd0,stroke:#c2185b,stroke-width:2px,color:#880e4f
    style LLM fill:#b3e5fc,stroke:#0277bd,stroke-width:2px,color:#01579b
    style OUT fill:#a5d6a7,stroke:#1b5e20,stroke-width:2px,color:#1b5e20
```

In [4]:
from sentence_transformers import SentenceTransformer
import faiss

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

chunk_texts = [c["text"] for c in CHUNKS]
embeddings = embedder.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

def retrieve(query, k=4):
    q = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")
    scores, ids = index.search(q, min(k, len(CHUNKS)))
    return [
        {**dict(CHUNKS[int(idx)]), "score": float(score)}
        for score, idx in zip(scores[0], ids[0])
    ]

print("Embedding dimension:", embeddings.shape[1])
print("FAISS vectors:", index.ntotal)
print([(r["title"], round(r["score"],3)) for r in retrieve("RAG و بازیابی اسناد", 3)])

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dimension: 384
FAISS vectors: 16
[('بازیابی تقویت\u200cشده با تولید', 0.538), ('بازیابی اطلاعات', 0.535), ('ارزیابی RAG', 0.369)]


## 5. Local LLM — Qwen2.5-0.5B-Instruct



In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if DEVICE == "cuda":
    llm = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
else:
    llm = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        trust_remote_code=True
    )

llm.eval()

def generate_text(prompt, max_new_tokens=180):
    messages = [
        {"role":"system","content":"شما یک دستیار پژوهشی دقیق هستید. فقط بر اساس context پاسخ دهید و اگر اطلاعات کافی نیست صریح بگویید اطلاعات کافی وجود ندارد."},
        {"role":"user","content":prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt")
    target_device = next(llm.parameters()).device
    inputs = {k:v.to(target_device) for k,v in inputs.items()}

    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(generate_text("در یک جمله توضیح بده RAG چیست.", 60))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

RAG مانند "R" (ریاضی)، "A" (آموزش)، "G" (گروه) یا "L" (لیست) است که در زبان فارسی به عنوان یک جمل


## 6. Query rewriting

The deployed agent uses the local LLM for real rewriting. PPO training uses a deterministic semantic expansion simulator so training remains fast on free Colab.

In [6]:
QUERY_EXPANSIONS = {
    "cnn":"CNN شبکه عصبی کانولوشنی تصویر کانولوشن pooling تشخیص شیء",
    "شبکه عصبی کانولوشنی":"CNN شبکه عصبی کانولوشنی تصویر کانولوشن pooling تشخیص شیء",
    "rag":"RAG retrieval augmented generation بازیابی اسناد context مدل زبانی",
    "ترنسفورمر":"Transformer attention BERT GPT NLP معماری ترنسفورمر",
    "ppo":"PPO Proximal Policy Optimization policy gradient clip reinforcement learning",
    "توصیه‌گر":"سیستم توصیه گر collaborative filtering content based hybrid",
    "embedding":"embedding بردار شباهت معنایی multilingual FAISS",
    "بازیابی":"information retrieval embedding similarity FAISS precision recall",
    "hallucination":"hallucination grounding faithfulness citation مدل زبانی",
    "ارزیابی":"RAG evaluation Recall Precision MRR faithfulness correctness",
}

def heuristic_rewrite(question):
    q = question.strip()
    additions = []
    ql = q.lower()
    for key, expansion in QUERY_EXPANSIONS.items():
        if key.lower() in ql:
            additions.append(expansion)
    if not additions:
        additions.append("تعریف مفاهیم کاربردها روش پیاده سازی مثال")
    return q + " | " + " | ".join(additions)

def llm_rewrite(question):
    prompt = f"""پرسش فارسی زیر را برای جست‌وجوی semantic در یک پایگاه دانش هوش مصنوعی بازنویسی کن.
فقط یک عبارت جست‌وجوی کوتاه و غنی از کلیدواژه‌ها خروجی بده.
پرسش: {question}"""
    out = generate_text(prompt, max_new_tokens=80).replace("\n"," ").strip()
    return out if out else heuristic_rewrite(question)

print(heuristic_rewrite("PPO چیست و چرا استفاده می‌شود؟"))

PPO چیست و چرا استفاده می‌شود؟ | PPO Proximal Policy Optimization policy gradient clip reinforcement learning


## 7. Retrieval and answer-quality proxy functions

## Agent Decision Flow

```mermaid
flowchart TD
    Start([🎯 Query arrives]) --> S1{📊 State<br/>has docs?}
    S1 -->|No| RET[🔍 Retrieve]
    S1 -->|Yes| S2{📈 Relevance<br/>high enough?}

    S2 -->|No| RW[✏️ Rewrite]
    S2 -->|Yes| ANS[💬 Answer]

    RW --> RA[🔁 Retrieve Again]
    RA --> ANS
    RET --> S3{Docs<br/>retrieved?}
    S3 -->|Yes| ANS
    S3 -->|No| RW

    ANS --> End([✅ Return Answer])

    style Start fill:#e1f5fe,stroke:#0277bd,color:#01579b
    style RET fill:#c8e6c9,stroke:#2e7d32,color:#1b5e20
    style RW fill:#fff9c4,stroke:#f9a825,color:#f57f17
    style RA fill:#b2dfdb,stroke:#00796b,color:#004d40
    style ANS fill:#d1c4e9,stroke:#5e35b1,color:#311b92
    style End fill:#a5d6a7,stroke:#1b5e20,color:#1b5e20
```

In [7]:
def keyword_coverage(query, results):
    q = query.lower()
    matched = set()
    for r in results:
        for k in r["keywords"]:
            if k.lower() in q:
                matched.add(k.lower())
    return min(1.0, len(matched) / 3.0)

def retrieval_features(question, results):
    scores = [r["score"] for r in results]
    if not scores:
        return {"mean":0.0,"max":0.0,"gap":0.0,"coverage":0.0,"n":0}
    gap = scores[0] - scores[1] if len(scores) > 1 else scores[0]
    return {
        "mean":float(np.mean(scores)),
        "max":float(np.max(scores)),
        "gap":float(gap),
        "coverage":float(keyword_coverage(question, results)),
        "n":len(scores)
    }

def lexical_f1(answer, reference):
    a=set(re.findall(r"\w+", answer.lower()))
    b=set(re.findall(r"\w+", reference.lower()))
    if not a or not b: return 0.0
    p=len(a&b)/len(a); r=len(a&b)/len(b)
    return 2*p*r/(p+r) if p+r else 0.0

def context_faithfulness_proxy(answer, contexts):
    aw=set(re.findall(r"\w+", answer.lower()))
    cw=set(re.findall(r"\w+", " ".join(contexts).lower()))
    return len(aw&cw)/len(aw) if aw else 0.0

## 8. PPO routing environment

Actions:

- `0 = ANSWER`
- `1 = RETRIEVE`
- `2 = REWRITE`
- `3 = RETRIEVE_AFTER_REWRITE`

The state includes retrieval quality, action history, rewrite status, step count and retrieval improvement. This fixes the hidden-state weakness of the first prototype.

## PPO Training Loop

```mermaid
flowchart LR
    ENV[🌍 Environment<br/>Gymnasium] -->|state sₜ| POL[🧠 Policy π]
    POL -->|action aₜ| ENV
    ENV -->|reward rₜ| BUF[📦 Trajectory<br/>Buffer]
    BUF --> UPD[⚙️ PPO Update<br/>clip · entropy]
    UPD -->|new π| POL

    style ENV fill:#bbdefb,stroke:#1565c0,color:#0d47a1
    style POL fill:#ffe0b2,stroke:#ef6c00,color:#e65100
    style BUF fill:#f8bbd0,stroke:#c2185b,color:#880e4f
    style UPD fill:#d1c4e9,stroke:#5e35b1,color:#311b92
```

In [8]:
import gymnasium as gym
from gymnasium import spaces

ACTIONS = {
    0:"ANSWER",
    1:"RETRIEVE",
    2:"REWRITE",
    3:"RETRIEVE_AFTER_REWRITE"
}

TRAIN_SET = [
    ("شبکه عصبی کانولوشنی چیست؟","cnn"),
    ("کاربرد CNN در بینایی ماشین چیست؟","cnn"),
    ("ترنسفورمر چگونه از attention استفاده می‌کند؟","transformer"),
    ("تفاوت RAG با مدل زبانی معمولی چیست؟","rag"),
    ("چگونه یک سیستم RAG بسازیم؟","rag"),
    ("Embedding چیست و چه کاربردی در FAISS دارد؟","embeddings"),
    ("PPO چیست؟","ppo"),
    ("PPO با Q-Learning چه تفاوتی دارد؟","rl"),
    ("سیستم توصیه‌گر چگونه ساخته می‌شود؟","recommendation"),
    ("بینایی ماشین صنعتی چه کاربردهایی دارد؟","cv"),
    ("توهم مدل زبانی چیست؟","hallucination"),
    ("چگونه faithfulness در RAG را ارزیابی کنیم؟","evaluation"),
    ("پایگاه برداری چیست؟","vector-db"),
    ("پردازش زبان طبیعی چیست؟","nlp"),
    ("مدل زبانی بزرگ چیست؟","llm"),
]

def target_coverage(results, target_doc):
    if not results: return 0.0
    rel=[r for r in results if r["doc_id"]==target_doc]
    return float(max([r["score"] for r in rel], default=0.0))

class AdaptiveRAGEnv(gym.Env):
    metadata={"render_modes":[]}

    def __init__(self, questions, max_steps=5):
        super().__init__()
        self.questions=questions
        self.max_steps=max_steps
        # mean, max, gap, coverage, has_docs, has_rewrite, step, last_action, improvement
        self.observation_space=spaces.Box(
            low=np.array([-1,-1,-1,0,0,0,0,0,-1],dtype=np.float32),
            high=np.array([1,1,1,1,1,1,1,1,1],dtype=np.float32),
            dtype=np.float32
        )
        self.action_space=spaces.Discrete(4)

    def _obs(self):
        f=retrieval_features(self.question,self.results)
        return np.array([
            np.clip(f["mean"],-1,1), np.clip(f["max"],-1,1),
            np.clip(f["gap"],-1,1), f["coverage"],
            float(bool(self.results)), float(self.rewritten),
            self.step_count/self.max_steps,
            self.last_action/3 if self.last_action>=0 else 0.0,
            np.clip(self.improvement,-1,1)
        ],dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.question,self.target=self.questions[self.np_random.integers(len(self.questions))]
        self.results=[]
        self.rewritten=False
        self.rewrite_query=""
        self.step_count=0
        self.last_action=-1
        self.improvement=0.0
        return self._obs(),{}

    def step(self,action):
        action=int(action)
        self.step_count+=1
        reward=-0.03
        terminated=False
        truncated=self.step_count>=self.max_steps

        before=target_coverage(self.results,self.target)

        if action==1: # retrieve
            if self.results: reward-=0.35
            self.results=retrieve(self.question,4)
            after=target_coverage(self.results,self.target)
            reward += 0.8*max(0,after-before)+0.35*after

        elif action==2: # rewrite
            if self.rewritten:
                reward-=0.30
            else:
                self.rewrite_query=heuristic_rewrite(self.question)
                self.rewritten=True
                reward+=0.10

        elif action==3: # retrieve after rewrite
            if not self.rewritten:
                self.rewrite_query=heuristic_rewrite(self.question)
                self.rewritten=True
            self.results=retrieve(self.rewrite_query,4)
            after=target_coverage(self.results,self.target)
            self.improvement=after-before
            reward += 0.8*max(0,self.improvement)+0.35*after

        elif action==0: # answer
            if not self.results:
                reward-=0.80
            else:
                quality=target_coverage(self.results,self.target)
                reward+=0.45*quality
                if quality>=0.20 or self.step_count>=2:
                    reward+=0.60*quality
                    terminated=True

        self.last_action=action
        return self._obs(),float(reward),terminated,truncated,{"action_name":ACTIONS[action]}

env=AdaptiveRAGEnv(TRAIN_SET)
print("Observation shape:",env.observation_space.shape)
print("Actions:",ACTIONS)

Observation shape: (9,)
Actions: {0: 'ANSWER', 1: 'RETRIEVE', 2: 'REWRITE', 3: 'RETRIEVE_AFTER_REWRITE'}


## 9. Train PPO

PPO is small enough to train on CPU, leaving the GPU for embeddings and local LLM inference.

In [9]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

check_env(env,warn=True)

TOTAL_TIMESTEPS=6000

ppo=PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=128,
    batch_size=64,
    n_epochs=10,
    gamma=0.97,
    gae_lambda=0.95,
    clip_range=0.20,
    ent_coef=0.02,
    vf_coef=0.5,
    max_grad_norm=0.5,
    seed=SEED,
    verbose=1,
    device="cpu"
)

ppo.learn(total_timesteps=TOTAL_TIMESTEPS)
print("PPO training complete.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 4.2      |
|    ep_rew_mean     | 0.503    |
| time/              |          |
|    fps             | 55       |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 128      |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 4.11        |
|    ep_rew_mean          | 0.448       |
| time/                   |             |
|    fps                  | 53          |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 256         |
| train/                  |             |
|    approx_kl            | 0.004402282 |
|    clip_fraction        | 0           |
|    clip_range           | 0.2         |
|    entropy_loss   

## Semantic Retrieval Pipeline

```mermaid
flowchart LR
    Q[❓ Query] --> E[🔢 Embedder<br/>MiniLM-L12]
    E --> V[📐 384-d Vector]
    V --> S[🔎 FAISS<br/>Inner Product]
    IDX[(🗂️ Vector Index<br/>16 vectors)] --> S
    S --> T[🎯 Top-K=4]
    T --> R[📄 Ranked Docs]

    style Q fill:#bbdefb,stroke:#1565c0,color:#0d47a1
    style E fill:#ffe0b2,stroke:#ef6c00,color:#e65100
    style V fill:#fff9c4,stroke:#f9a825,color:#f57f17
    style S fill:#f8bbd0,stroke:#c2185b,color:#880e4f
    style IDX fill:#e1bee7,stroke:#6a1b9a,color:#4a148c
    style T fill:#b2dfdb,stroke:#00796b,color:#004d40
    style R fill:#c8e6c9,stroke:#2e7d32,color:#1b5e20
```

## 10. Inspect the learned policy

In [10]:
from collections import Counter

def run_policy(question,target_doc,deterministic=True):
    e=AdaptiveRAGEnv([(question,target_doc)],max_steps=5)
    obs,_=e.reset(seed=SEED)
    trace=[]
    total_reward=0.0
    for _ in range(e.max_steps):
        action,_=ppo.predict(obs,deterministic=deterministic)
        obs,reward,term,trunc,info=e.step(int(action))
        total_reward+=reward
        f=retrieval_features(e.question,e.results)
        trace.append({
            "step":e.step_count,
            "action":ACTIONS[int(action)],
            "reward":round(float(reward),3),
            "top_score":round(f["max"],3),
            "rewritten":e.rewritten
        })
        if term or trunc: break
    return trace,total_reward,e

counter=Counter()
for q,t in TRAIN_SET:
    trace,_,_=run_policy(q,t)
    for row in trace: counter[row["action"]]+=1
print("Action counts:")
for k,v in counter.items(): print(f"{k:28s}{v}")

Action counts:
RETRIEVE_AFTER_REWRITE      75


## 11. Local LLM answer generation and Agentic RAG

In [11]:
def build_context(results):
    return "\n\n".join([f"[{i+1}] {r['title']}\n{r['text']}" for i,r in enumerate(results)])

def answer_with_context(question,results):
    context=build_context(results)
    prompt=f"""به پرسش زیر فقط بر اساس context پاسخ بده.

پرسش:
{question}

context:
{context}

قواعد:
- پاسخ فارسی و دقیق باشد.
- اگر context کافی نیست، صریح بگو اطلاعات کافی وجود ندارد.
- از ساختن واقعیت خارج از context خودداری کن.
"""
    return generate_text(prompt,max_new_tokens=180)

def adaptive_agent(question,max_steps=5):
    results=[]
    current_query=question
    rewritten=False
    trace=[]

    for step in range(max_steps):
        f=retrieval_features(question,results)
        obs=np.array([
            np.clip(f["mean"],-1,1),np.clip(f["max"],-1,1),
            np.clip(f["gap"],-1,1),f["coverage"],
            float(bool(results)),float(rewritten),
            step/max_steps,0.0,0.0
        ],dtype=np.float32)

        action,_=ppo.predict(obs,deterministic=True)
        action=int(action)

        if action==0:
            trace.append({"step":step+1,"action":"ANSWER","query":current_query,
                          "top_score":round(results[0]["score"],3) if results else None})
            break

        if action==1:
            if results:
                # guard against redundant retrieval loops
                trace.append({"step":step+1,"action":"ANSWER","query":current_query,
                              "top_score":round(results[0]["score"],3)})
                break
            results=retrieve(current_query,4)
            name="RETRIEVE"

        elif action==2:
            current_query=llm_rewrite(current_query)
            rewritten=True
            name="REWRITE"

        else:
            if not rewritten:
                current_query=llm_rewrite(current_query)
                rewritten=True
            results=retrieve(current_query,4)
            name="RETRIEVE_AFTER_REWRITE"

        trace.append({"step":step+1,"action":name,"query":current_query,
                      "top_score":round(results[0]["score"],3) if results else None})

    if not results:
        results=retrieve(current_query,4)
        trace.append({"step":len(trace)+1,"action":"FALLBACK_RETRIEVE",
                      "query":current_query,"top_score":round(results[0]["score"],3)})

    answer=answer_with_context(question,results)
    trace.append({"step":len(trace)+1,"action":"ANSWER","query":current_query,
                  "top_score":round(results[0]["score"],3)})
    return {"question":question,"rewritten_query":current_query if rewritten else None,
            "results":results,"answer":answer,"trace":trace}

demo=adaptive_agent("تفاوت RAG و مدل زبانی معمولی چیست؟")
print("ROUTE:")
for x in demo["trace"]: print(x)
print("\nANSWER:\n",demo["answer"])

ROUTE:
{'step': 1, 'action': 'RETRIEVE_AFTER_REWRITE', 'query': 'RAG (Recurrent Attention-based Language Generation) و زبانی معمولی (Transformer) تفاوت دارد که به طور کلی به شکل زیر است:  1. **RAG**:     - **Recurrent Attention**: این روش است که از ساختار زبانی معمولی به روش زبان', 'top_score': 0.557}
{'step': 2, 'action': 'RETRIEVE_AFTER_REWRITE', 'query': 'RAG (Recurrent Attention-based Language Generation) و زبانی معمولی (Transformer) تفاوت دارد که به طور کلی به شکل زیر است:  1. **RAG**:     - **Recurrent Attention**: این روش است که از ساختار زبانی معمولی به روش زبان', 'top_score': 0.557}
{'step': 3, 'action': 'RETRIEVE_AFTER_REWRITE', 'query': 'RAG (Recurrent Attention-based Language Generation) و زبانی معمولی (Transformer) تفاوت دارد که به طور کلی به شکل زیر است:  1. **RAG**:     - **Recurrent Attention**: این روش است که از ساختار زبانی معمولی به روش زبان', 'top_score': 0.557}
{'step': 4, 'action': 'RETRIEVE_AFTER_REWRITE', 'query': 'RAG (Recurrent Attention-based Language Generat

## 12. Baseline vs Adaptive Agent evaluation


In [12]:
EVAL_SET=[
 {"q":"شبکه عصبی کانولوشنی چیست؟","target":"cnn","ref":"CNN یک شبکه عصبی برای پردازش تصویر است که از کانولوشن و pooling استفاده می‌کند."},
 {"q":"RAG چیست و چگونه کار می‌کند؟","target":"rag","ref":"RAG ابتدا اسناد مرتبط را بازیابی می‌کند و سپس مدل زبانی با استفاده از context پاسخ می‌دهد."},
 {"q":"PPO چیست؟","target":"ppo","ref":"PPO یک الگوریتم policy-gradient در یادگیری تقویتی است که تغییرات policy را محدود می‌کند."},
 {"q":"Embedding در جست‌وجوی معنایی چه نقشی دارد؟","target":"embeddings","ref":"Embedding متن را به بردار تبدیل می‌کند تا شباهت معنایی برای جست‌وجو محاسبه شود."},
 {"q":"سیستم توصیه‌گر چگونه کار می‌کند؟","target":"recommendation","ref":"سیستم توصیه‌گر با استفاده از تعاملات کاربر و ویژگی اقلام پیشنهاد شخصی‌سازی‌شده تولید می‌کند."},
 {"q":"توهم مدل زبانی چیست؟","target":"hallucination","ref":"Hallucination تولید اطلاعات نادرست یا بدون پشتوانه توسط مدل زبانی است."},
 {"q":"ترنسفورمر چه کاربردی در NLP دارد؟","target":"transformer","ref":"Transformer با attention برای مدل‌سازی وابستگی‌های توالی و کاربردهای NLP استفاده می‌شود."},
 {"q":"FAISS چیست؟","target":"vector-db","ref":"FAISS کتابخانه‌ای برای ذخیره و جست‌وجوی سریع بردارها و similarity search است."},
]

def baseline_rag(q):
    rs=retrieve(q,4)
    return rs,answer_with_context(q,rs)

rows=[]
for item in EVAL_SET:
    q,target,ref=item["q"],item["target"],item["ref"]

    t0=time.time()
    br,ba=baseline_rag(q)
    bt=time.time()-t0

    t0=time.time()
    aa=adaptive_agent(q)
    at=time.time()-t0

    rows.append({
        "question":q,
        "baseline_hit":int(any(r["doc_id"]==target for r in br)),
        "agent_hit":int(any(r["doc_id"]==target for r in aa["results"])),
        "baseline_answer_f1":round(lexical_f1(ba,ref),3),
        "agent_answer_f1":round(lexical_f1(aa["answer"],ref),3),
        "baseline_faithfulness":round(context_faithfulness_proxy(ba,[r["text"] for r in br]),3),
        "agent_faithfulness":round(context_faithfulness_proxy(aa["answer"],[r["text"] for r in aa["results"]]),3),
        "agent_steps":len(aa["trace"]),
        "rewritten":int(aa["rewritten_query"] is not None),
        "baseline_time":round(bt,2),
        "agent_time":round(at,2)
    })

import pandas as pd
df=pd.DataFrame(rows)
display(df)

print("\n=== Aggregate ===")
for c in ["baseline_hit","agent_hit","baseline_answer_f1","agent_answer_f1",
          "baseline_faithfulness","agent_faithfulness","agent_steps","rewritten"]:
    print(f"{c:28s}: {df[c].mean():.3f}")

,question,baseline_hit,agent_hit,baseline_answer_f1,agent_answer_f1,baseline_faithfulness,agent_faithfulness,agent_steps,rewritten,baseline_time,agent_time
0,شبکه عصبی کانولوشنی چیست؟,1,1,0.490,0.490,0.970,0.970,6,1,30.58,44.85
1,RAG چیست و چگونه کار می‌کند؟,1,1,0.540,0.128,0.756,0.724,6,1,34.22,48.43
2,PPO چیست؟,1,1,0.522,0.522,1.000,1.000,6,1,18.75,21.81
3,Embedding در جست‌وجوی معنایی چه نقشی دارد؟,1,1,0.464,0.457,0.792,0.870,6,1,33.95,43.53
4,سیستم توصیه‌گر چگونه کار می‌کند؟,1,1,0.923,0.923,0.952,0.905,6,1,17.28,31.46
5,توهم مدل زبانی چیست؟,0,0,0.080,0.080,0.357,0.357,6,1,13.86,27.82
6,ترنسفورمر چه کاربردی در NLP دارد؟,1,0,0.176,0.232,0.526,0.778,6,1,16.16,39.31
7,FAISS چیست؟,0,1,0.143,0.290,0.465,0.837,6,1,31.57,50.13



=== Aggregate ===
baseline_hit                : 0.750
agent_hit                   : 0.750
baseline_answer_f1          : 0.417
agent_answer_f1             : 0.390
baseline_faithfulness       : 0.727
agent_faithfulness          : 0.805
agent_steps                 : 6.000
rewritten                   : 1.000


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Baseline vs Adaptive Comparison

```mermaid
flowchart TB
    subgraph FIXED["🔒 Fixed RAG (Baseline)"]
        direction LR
        Q1([Query]) --> R1[🔍 Retrieve<br/>always] --> A1[💬 Answer] --> O1([Output])
    end

    subgraph ADAPT["🎯 Adaptive Agentic RAG (Ours)"]
        direction LR
        Q2([Query]) --> P{{🧠 PPO}}
        P -->|R| R2[🔍 Retrieve]
        P -->|W| W2[✏️ Rewrite]
        P -->|A| A2[💬 Answer]
        R2 --> A2
        W2 --> A2
        A2 --> O2([Output])
    end

    style FIXED fill:#eceff1,stroke:#455a64,color:#263238
    style ADAPT fill:#e8f5e9,stroke:#2e7d32,color:#1b5e20
    style P fill:#ffe0b2,stroke:#ef6c00,color:#e65100
    style O1 fill:#cfd8dc,stroke:#37474f,color:#263238
    style O2 fill:#a5d6a7,stroke:#1b5e20,color:#1b5e20
```

## 13. Routing diagnostics

In [13]:
print("=== ROUTING TRACES ===")
for item in EVAL_SET:
    a=adaptive_agent(item["q"])
    route=" → ".join(x["action"] for x in a["trace"])
    print("\nQ:",item["q"])
    print("Route:",route)
    if a["rewritten_query"]:
        print("Rewrite:",a["rewritten_query"])
    print("Top docs:",[(r["title"],round(r["score"],3)) for r in a["results"][:3]])

=== ROUTING TRACES ===

Q: شبکه عصبی کانولوشنی چیست؟
Route: RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → ANSWER
Rewrite: شبکه عصبی کانولوشنی یک نرم‌افزار مرتبط با شبکه‌های عصبی است که به عنوان یک نرم‌افزار مدل‌های عصبی و تحلیل‌های عصبی مانند کانولوشنی، یکی ا
Top docs: [('شبکه عصبی کانولوشنی', 0.628), ('یادگیری عمیق', 0.602), ('ترنسفورمر', 0.368)]

Q: RAG چیست و چگونه کار می‌کند؟
Route: RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → ANSWER
Rewrite: RAG (Reinforcement Learning Agent) است که به عنوان یک دستیار پژوهشی دقیق، به طور کلی برای تحلیل و تحلیل مورد نظریه‌ای از داده‌های موجود استفاده می‌کند. این دستیار
Top docs: [('پایگاه برداری', 0.553), ('عامل\u200cهای هوشمند', 0.544), ('ارزیابی RAG', 0.469)]

Q: PPO چیست؟
Route: RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → RETRIEVE_AFTER_REWRITE → RETRIEVE_AFT

## End-to-End Agentic RAG Flow

```mermaid
sequenceDiagram
    autonumber
    actor U as 👤 User
    participant R as 🧠 PPO Router
    participant K as 🗂️ FAISS KB
    participant L as 🤖 Local LLM

    U->>R: Persian query
    R->>R: Compute state
    R->>K: Action: RETRIEVE
    K-->>R: Top-4 docs + scores
    R->>R: Update state
    R->>L: Action: ANSWER (with context)
    L-->>R: Persian answer
    R-->>U: Final response

    Note over R,K: Optional: REWRITE if<br/>retrieval quality is low
```

## 14. Save artifacts for GitHub

In [14]:
os.makedirs("artifacts",exist_ok=True)

ppo.save("artifacts/adaptive_rag_ppo")

metadata={
    "seed":SEED,
    "embedding_model":EMBED_MODEL,
    "llm_model":MODEL_ID,
    "total_timesteps":TOTAL_TIMESTEPS,
    "actions":ACTIONS,
    "documents":len(DOCUMENTS),
    "chunks":len(CHUNKS),
    "eval_questions":len(EVAL_SET),
    "device":DEVICE
}

with open("artifacts/experiment_config.json","w",encoding="utf-8") as f:
    json.dump(metadata,f,ensure_ascii=False,indent=2)

df.to_csv("artifacts/evaluation.csv",index=False,encoding="utf-8-sig")

print("Saved:")
print("artifacts/adaptive_rag_ppo.zip")
print("artifacts/experiment_config.json")
print("artifacts/evaluation.csv")

Saved:
artifacts/adaptive_rag_ppo.zip
artifacts/experiment_config.json
artifacts/evaluation.csv


---

# 🌟 Enjoyed This Notebook?

If this project helped you or inspired your own work, please consider:

### ⭐ Give it a Star
A star on GitHub helps others discover this project and motivates continued development.

### 🐛 Report Issues or Suggest Ideas
Found a bug? Have an idea for improvement? Open an issue — contributions are always welcome.

### 📢 Share It
Share this notebook with your network — on LinkedIn, Twitter/X, or with your team.

### 🔗 Let's Connect

| Platform | Link |
|---|---|
| 🐙 **GitHub** | [github.com/omid-sakaki-ghazvini](https://github.com/omid-sakaki-ghazvini) |
| 💼 **LinkedIn** | [linkedin.com/in/omid-sakaki-ghazvini](https://www.linkedin.com/in/omid-sakaki-ghazvini-378687217) |
| 📊 **Kaggle** | [kaggle.com/omidsakaki1370](https://www.kaggle.com/omidsakaki1370) |
| 🌐 **Website** | [omidsakaki.ir](https://omidsakaki.ir/) |

---

<p align="center">
  <b>Built with ❤️ by Omid Sakaki Ghazvini</b><br/>
  <i>AI Engineer · Computer Vision · NLP · Reinforcement Learning</i>
</p>

<p align="center">
  <a href="https://github.com/omid-sakaki-ghazvini">
    <img src="https://img.shields.io/badge/GitHub-Follow-181717?style=for-the-badge&logo=github" />
  </a>
  <a href="https://www.linkedin.com/in/omid-sakaki-ghazvini-378687217">
    <img src="https://img.shields.io/badge/LinkedIn-Connect-0A66C2?style=for-the-badge&logo=linkedin" />
  </a>
  <a href="https://www.kaggle.com/omidsakaki1370">
    <img src="https://img.shields.io/badge/Kaggle-Follow-20BEFF?style=for-the-badge&logo=kaggle" />
  </a>
  <a href="https://omidsakaki.ir/">
    <img src="https://img.shields.io/badge/Website-Visit-FF5722?style=for-the-badge&logo=googlechrome&logoColor=white" />
  </a>
</p>